# 03. ????????? ??????????????? ??????

**????.** ????????????? ????????? ?????? ??? ?????? fusion-???????: log branch + KPI branch + modality masks + fused heads. ??????? ???????? ???????? ???????????, ?????? ? ????????; ??????? ?????????? ???????? ????????? ? `dashboard.train_multimodal_fusion`.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

from dashboard.final_data_pipeline import PROJECT_ROOT, artifact_paths
from dashboard.export_dashboard_bundle import export_bundle

ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
PATHS = artifact_paths(ARTIFACT_DIR)
RUN_ID = "final_multimodal_strict_intensity_gap10"
MODEL_RUN_DIR = PATHS.model_dir / RUN_ID
RUN_TRAIN = os.environ.get("RUN_FINAL_TRAIN", "0") == "1"

train_command = [
    sys.executable,
    "-m", "dashboard.train_multimodal_fusion",
    "--contract", str(PATHS.contract_path),
    "--artifact-dir", str(ARTIFACT_DIR),
    "--run-id", RUN_ID,
    "--seeds", "42,43,44",
]
print("??????? ????????:")
print(" ".join(map(str, train_command)))
print("RUN_FINAL_TRAIN=", RUN_TRAIN)


??????? ????????:
C:\Users\ipopi\AppData\Local\Programs\Python\Python313\python.exe -m dashboard.train_multimodal_fusion --contract C:\Main\DGTU\ВКР\Diploma\dashboard\artifacts\contract\strict_intensity_contract.csv.gz --artifact-dir C:\Main\DGTU\ВКР\Diploma\dashboard\artifacts --run-id final_multimodal_strict_intensity_gap10 --seeds 42,43,44
RUN_FINAL_TRAIN= False


## 1. ????????? ??????????

?????? ???????? ?? ?????? ????? ????? ? KPI-????????, ?? ? ????? ??????????? ????????????. ??? ?????: ?????????? ???? ??? KPI ? ????????? ??????????????? ????, ? ?? ???? ? ?????????. ???????? score ??? ???????? ???????? ?? intensity-head, ?????? ??? ???????? ??????? ?????? ???????? ?????????? ????? ?? ???????, ? ?? ?????? ???????? ???? ????.


In [2]:
architecture = pd.DataFrame([
    {"????": "Log branch", "????": "???????? top log events ?? ???????? ????"},
    {"????": "KPI branch", "????": "???????? ?????????????????? KPI ?? ?????????? ????"},
    {"????": "Modality masks", "????": "???????? ??????, ????? ??????????? ????????"},
    {"????": "Fusion block", "????": "?????????? log/KPI ????????????? ? ???? hidden state"},
    {"????": "Binary heads", "????": "??????? warning/risk ????"},
    {"????": "Intensity heads", "????": "??????? ?????????? ????? ?? lead time"},
])
display(architecture)


,????
0,???????? top log events ?? ???????? ????
1,???????? ?????????????????? KPI ?? ?????????? ...
2,"???????? ??????, ????? ??????????? ????????"
3,?????????? log/KPI ????????????? ? ???? hidden...
4,??????? warning/risk ????
5,??????? ?????????? ????? ?? lead time


**????????? ?????.** ????????? ?????? ???????? ?????? ???????????????: ???? fused-??????? ???????? ?? ???? ?????? ? ????? ???????????. Log-only ? KPI-only ???????????? ?????? ??? baseline/???????????, ? ?? ??? ??????????? ???????.


## 2. ????????

?? ????????? notebook ?? ????????? ??????? ???????? ??? ?????? ????????. ??? ??????? ???????????????? train-run ????? ????????? notebook ??? ??????? ???? ? ?????????? ????????? `RUN_FINAL_TRAIN=1`. ?? ?????? ? CUDA ????? ???????? `--device cuda --text-backend embeddinggemma`.


In [3]:
if RUN_TRAIN:
    subprocess.run(train_command, check=True)
elif MODEL_RUN_DIR.exists():
    print(f"?????? ??????? train-run: {MODEL_RUN_DIR.relative_to(PROJECT_ROOT)}")
else:
    print("Train-run ???? ?? ??????. ????????? ??????? ???????? ???? ??? ?????????? RUN_FINAL_TRAIN=1.")


Train-run ???? ?? ??????. ????????? ??????? ???????? ???? ??? ?????????? RUN_FINAL_TRAIN=1.


**????????? ?????.** ???????? ???????? ? ?????? Python-????????. ??? ?????? ??????? ??????????: ???? ?????? ?????? ??????????, preprocessing, alert-layer ? replay-????????? ???????? ???? ??.


## 3. ??????? ? ????????? ? baseline

????? ???????? ?????? ??????? ?? `dashboard/artifacts/model/<run_id>/`. ??????? ?????????: `warning_intensity_score` ? `state_score` ?????? baseline ?? ??? ?? strict/intensity contract.


In [4]:
metrics_path = MODEL_RUN_DIR / "ensemble_metrics.csv"
baseline_path = MODEL_RUN_DIR / "baseline_comparison.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    focus = metrics.loc[
        metrics["score"].isin(["warning_intensity_score", "risk_intensity_score", "state_score"]),
        ["target", "model", "score", "average_precision", "roc_auc", "precision", "recall", "f1"],
    ]
    display(focus)
else:
    print("??????? ?????? ???? ?? ???????.")

if baseline_path.exists():
    baseline = pd.read_csv(baseline_path)
    display(baseline.sort_values(["target", "average_precision"], ascending=[True, False]))
else:
    print("Baseline comparison train-run ???? ?? ??????.")


??????? ?????? ???? ?? ???????.
Baseline comparison train-run ???? ?? ??????.


**????????? ?????.** Primary-?????? ?????? ? ranking/alert usefulness, ? ?? ???????? ?????????? ????????? ???????. Risk-channel ???????? ???????????????: ???? ?? ?????? warning-intensity, ??????? ?????? ?????????? ??? ??????.


## 4. ??????? ??? ????????

??????? ?? ??????? ?????? ? ?? ?????? ?????? ????????????????? ???????. ?? ?????????? ?????? ?????????? bundle: replay-????, score, alert-state, ??????? run, baseline comparison ? model report.


In [5]:
if (MODEL_RUN_DIR / "ensemble_predictions.csv").exists() and PATHS.contract_path.exists():
    report = export_bundle(model_run_dir=MODEL_RUN_DIR, contract_path=PATHS.contract_path, out_dir=PATHS.bundle_dir)
    print(json.dumps({
        "bundle_dir": str(PATHS.bundle_dir.relative_to(PROJECT_ROOT)),
        "model_name": report.get("model_name"),
        "score_source": report.get("score_source"),
    }, ensure_ascii=False, indent=2))
else:
    print("Bundle ?? ?????????????: ??????? ????? train-run ?????? ? contract ?? 02.")


Bundle ?? ?????????????: ??????? ????? train-run ?????? ? contract ?? 02.


**???????? handoff.** ??? ??????? ?????????? ????? ?????? ????? ?? `dashboard/artifacts/bundle/`. ???????: `streamlit run dashboard/mm_dashboard.py`. ???? bundle ???????????, ??????? ?????????? ??????? ?????????? ? ?? ???????? proxy-scoring.
